In [516]:
import random
from fractions import Fraction
from typing import List, Tuple
import pandas as pd
from datasets import Dataset

import pandas as pd 
import os 

from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset

import seaborn as sns

from datasets import load_dataset, load_from_disk
import numpy as np
import json

from utils.corrupt_num import *
from utils.chunk_r import *

## Evals - How to Scale:
1. number of operands
2. new operations

In [169]:
nums = [ 47, 2, 42, 9, 62, 20, 64 ]
target = 228
question = f"Using the numbers {nums}, create an equation that equals {target}. You can use basic arithmetic operations (+, -, *, /) and each number can only be used once"

In [170]:
tokenizer = AutoTokenizer.from_pretrained("zwhe99/DeepMath-1.5B")

In [171]:
conversations = [
    {"role": "system", "content": "Please reason and answer the following question. You should return the final answer in <answer></answer> tags, for example <answer>(1 + 2) / 3</answer>."},
    {"role": "user", "content": question}
]
prompt = tokenizer.apply_chat_template(
    conversations,
    tokenize=False,
    add_generation_prompt=True
)

In [172]:
print(prompt)

<｜begin▁of▁sentence｜>Please reason and answer the following question. You should return the final answer in <answer></answer> tags, for example <answer>(1 + 2) / 3</answer>.<｜User｜>Using the numbers [47, 2, 42, 9, 62, 20, 64], create an equation that equals 228. You can use basic arithmetic operations (+, -, *, /) and each number can only be used once<｜Assistant｜><think>



In [ ]:
"Please reason and answer the following question. At the end, you should return the final answer in <answer></answer> tags, for example <answer>(1 + 2) / 3</answer>. Using the numbers {nums}, create an equation that equals {target}. You can use basic arithmetic operations (+, -, *, /) in the expression and each number can only be used once."

In [173]:
47 + 62 + 9 + 64 + 42/2

203.0

# Check Eval Dataset

In [828]:
cd /home/al2644/research/codebase/reasoning/perturb-r/

/home/al2644/research/codebase/reasoning/perturb-r


In [816]:
path = "./data/train/countdown_level45_20k/train.parquet"
df = pd.read_parquet(path)

In [819]:
df["prompt"][0]

array([{'content': 'Please reason and answer the following question. Using the numbers [50, 34, 97, 68], create an equation that equals 181. You can only use basic arithmetic operations (+, -, *, /) in the expression and each number should be used exactly once. You should report the answer equation expression in <answer> </answer> tags, for example <answer> (1 + 2) / 3 </answer>.', 'role': 'user'}],
      dtype=object)

In [823]:
def format_dataset(row, tokenizer = tokenizer):
    problem = row['problem']
    response = row['response']
    conversation = [
        {"role": "user", "content": problem},
        {"role": "assistant", "content": response}
    ]

    return tokenizer.apply_chat_template(conversation, tokenize=False)

In [383]:
result = format_dataset(row)

# Reasoning Benchmark Eval

In [837]:
root = "./results/countdown/benchmark/"
tokenizer = AutoTokenizer.from_pretrained('Qwen/Qwen2.5-3B')
for fname in os.listdir(root):
    if 'correct' not in fname:
        print(fname)
        df = pd.read_pickle(os.path.join(root, fname))
        print(f"Accuracy: {df['correct'].mean()}")

# model specific
df = pd.read_pickle(os.path.join(root, 'Qwen2.5-3B-grpo-countdown-level4-5-1epoch.pickle'))

Qwen2.5-3B-distill-countdown-level3-4-1epoch.pickle
Accuracy: 0.24214285714285713
Qwen2.5-3B-distill-countdown-level3-4-3epoch.pickle
Accuracy: 0.24714285714285714
Qwen2.5-3B-grpo-countdown-level4-5-1epoch.pickle
Accuracy: 0.35


In [612]:
def groupby_level (df):
    stats = df.groupby('level')['correct'].mean()
    return stats

In [835]:
groupby_level(df)

level
4     0.890
5     0.550
6     0.205
7     0.050
9     0.000
11    0.000
13    0.000
Name: correct, dtype: float64

In [838]:
groupby_level(df)

level
4     0.935
5     0.745
6     0.450
7     0.310
9     0.010
11    0.000
13    0.000
Name: correct, dtype: float64

# Corrupted Numbers

In [840]:
pwd

'/home/al2644/research/codebase/reasoning/perturb-r'

In [841]:
root = "./results/countdown/corrupt_numbers/"

def has_answer (response):
    if "<answer>" in response:
        return 1.
    else:
        return 0.
    
for fname in os.listdir(root):
    if 'still_correct' not in fname:
        print(fname)
        df = pd.read_pickle(os.path.join(root, fname))
        print(f"Accuracy: {df['still_correct'].mean()}")

Qwen2.5-3B-distill-countdown-level3-4-1epoch.pickle
Accuracy: 0.5202702702702703
Qwen2.5-3B-grpo-countdown-level4-5-1epoch.pickle
Accuracy: 0.5934684684684685


In [842]:
df = pd.read_pickle(os.path.join(root, "Qwen2.5-3B-grpo-countdown-level4-5-1epoch.pickle"))


In [844]:
df.groupby("start_pos")["still_correct"].mean()

start_pos
0.00    0.184685
0.25    0.720721
0.50    0.752252
0.75    0.716216
Name: still_correct, dtype: float64

In [854]:
df.groupby("start_pos")["if_answer"].mean()

start_pos
0.00    0.409910
0.25    0.765766
0.50    0.788288
0.75    0.995495
Name: if_answer, dtype: float64

In [858]:
example = df[(df["start_pos"] == 0.5) & (df["still_correct"] == 0.) & (df["if_answer"] == True)].iloc[0]

In [859]:
print(example["prompt"])

<|im_start|>system
You are a helpful assistant.<|im_end|>
<|im_start|>user
Please reason and answer the following question. Using the numbers [13, 10, 27, 92, 31], create an equation that equals 742. You can only use basic arithmetic operations (+, -, *, /) in the expression and each number should be used exactly once. You should report the answer equation expression in <answer> </answer> tags, for example <answer> (1 + 2) / 3 </answer>.<|im_end|>
<|im_start|>assistant
<think>


Okay, let's see. I need to use the numbers 13, 10, 27, 92, and 31 exactly once each, with basic operations to make 742. Hmm. Let me think. First, maybe I should try combining some numbers to get closer to 742. Since 742 is a relatively large number, maybe multiplication is involved here. Let me check the numbers: 92 is


the largest, then 31, 27, 10, 13.


Let me think about possible multiplications. If I multiply 92 by something, maybe? Let's see. 92 * 8 is 736, which is close to 742. But how do I get 8 from t

In [860]:
print(example["post_corruption_response"])

 

Alternatively, think of 92 * 10 = 920. Then 920 - 27 = 893. 893 - 13 = 880. 880 + 31 = 911. No, that's not helpful. 

Alternatively, 92 * 27 = 2484. Then 2484 - 31 = 2453. 2453 - 13 = 2440. 2440 + 10 = 2450. Not 742. 

Alternatively, 92 * 13 = 1204. Then 1204 - 31 = 1173. 1173 - 27 = 1146. 1146 - 10 = 1136. That's way over. 

Alternatively, 92 * 31 = 2852. Then 2852 - 27 = 2825. 2825 - 13 = 2812. 2812 - 10 = 2802. No. 

Alternatively, 31 * 10 = 310. Then 310 + 27 = 337. 337 + 92 = 429. 429 + 13 = 442. No. 

Wait, maybe 27 * 13 = 351. Then 351 * 2 = 702. Then 702 + 40 = 742. How to get 40 from 92 and 31. 92 - 31 = 61. Not 40. 92 - 31 is 61, then 61 - 10 = 51. No. 

Alternatively, 27 * 10 = 270. Then 270 + 92 = 362. 362 * something? 362 * 2 is 724. Then 724 + 18 = 742. How to get 18 from 13 and 31. 31 - 13 = 18. Oh! Wait! That's possible. 

So let's see: (27 * 10) + 92 = 362. Then 362 + (31 - 13) = 362 + 18 = 380. No, that's not. Wait, but if I do 27 * 10 + 92 + (31 -13) = 362 + 92 + 

In [695]:
wrong_answer[['level', 'type']].value_counts()

level  type         
5      post_</think>    80
6      post_</think>    21
5      pre_</think>     17
6      pre_</think>      5
Name: count, dtype: int64

# Distractor Injection

In [796]:
root = "./results/countdown/inject_distractor/"

def has_answer (response):
    if "<answer>" in response:
        return 1.
    else:
        return 0.
    
for fname in os.listdir(root):
    print(fname)
    df = pd.read_pickle(os.path.join(root, fname))
    print(f"Original Accuracy: {df['original_correct'].mean()}")
    print(f"Distractor Accuracy: {df['distractor_correct'].mean()}")

df = pd.read_pickle(os.path.join(root, "Qwen2.5-3B-grpo-countdown-level4-5-1epoch.pickle"))

Qwen2.5-3B-distill-countdown-level3-4-1epoch.pickle
Original Accuracy: 0.3626237623762376
Distractor Accuracy: 0.5006188118811881
Qwen2.5-3B-grpo-countdown-level4-5-1epoch.pickle
Original Accuracy: 0.46163366336633666
Distractor Accuracy: 0.47277227722772275


In [797]:
df = pd.read_pickle(os.path.join(root, "Qwen2.5-3B-distill-countdown-level3-4-1epoch.pickle"))
df.groupby("prefix_ratio")[["original_correct", "distractor_correct"]].mean()

,original_correct,distractor_correct
prefix_ratio,,
0.00,0.047030,0.816832
0.25,0.544554,0.292079
0.50,0.462871,0.400990
0.75,0.396040,0.492574


In [798]:
df = pd.read_pickle(os.path.join(root, "Qwen2.5-3B-grpo-countdown-level4-5-1epoch.pickle"))
df.groupby("prefix_ratio")[["original_correct", "distractor_correct"]].mean()

,original_correct,distractor_correct
prefix_ratio,,
0.00,0.009901,0.938119
0.25,0.670792,0.252475
0.50,0.613861,0.309406
0.75,0.551980,0.391089


In [799]:
df

,nums,target,example_solution,problem,level,task,original_response,reasoning_chunks,prefix_ratio,distractor_ratio,distract_nums,distract_target,distract_reasoning,prompt,post_distraction_response,original_correct,distractor_correct
0,"[50, 77, 64, 67, 73]",331,(((73+50)+64)+(67+77)),Please reason and answer the following questio...,5,countdown,"<think>\nOkay, let's see. I need to use each o...","[<think>\nOkay, let's see. I need to use each ...",0.0,0.25,"[86, 91, 87, 68, 77, 66]",165,"<think>\nOkay, let's see. I need to use the nu...",<|im_start|>system\nYou are a helpful assistan...,"Not sure. Alternatively, 87 - 66 = 21. 91 - 7...",0.0,1.0
1,"[50, 77, 64, 67, 73]",331,(((73+50)+64)+(67+77)),Please reason and answer the following questio...,5,countdown,"<think>\nOkay, let's see. I need to use each o...","[<think>\nOkay, let's see. I need to use each ...",0.0,0.5,"[20, 73, 85, 82, 33]",89,"<think>\nOkay, let's see. I need to use the nu...",<|im_start|>system\nYou are a helpful assistan...,33 + 85 is 118. 118 - 82 is 36. 36 +73 is 109...,0.0,1.0
2,"[50, 77, 64, 67, 73]",331,(((73+50)+64)+(67+77)),Please reason and answer the following questio...,5,countdown,"<think>\nOkay, let's see. I need to use each o...","[<think>\nOkay, let's see. I need to use each ...",0.0,0.75,"[20, 73, 85, 82, 33]",89,"<think>\nOkay, let's see. I need to use the nu...",<|im_start|>system\nYou are a helpful assistan...,So the answer would be that expression. So th...,0.0,1.0
3,"[50, 77, 64, 67, 73]",331,(((73+50)+64)+(67+77)),Please reason and answer the following questio...,5,countdown,"<think>\nOkay, let's see. I need to use each o...","[<think>\nOkay, let's see. I need to use each ...",0.0,1.0,"[74, 51, 64, 31, 48]",104,"<think>\nOkay, let's see. I need to use the nu...",<|im_start|>system\nYou are a helpful assistan...,The answer should be in <answer> tags. So <an...,0.0,1.0
4,"[50, 77, 64, 67, 73]",331,(((73+50)+64)+(67+77)),Please reason and answer the following questio...,5,countdown,"<think>\nOkay, let's see. I need to use each o...","[<think>\nOkay, let's see. I need to use each ...",0.25,0.25,"[25, 87, 42, 34, 57]",63,"<think>\nOkay, let's see. I need to use each o...",<|im_start|>system\nYou are a helpful assistan...,"87 - 57 is 30, then 30 - 25 is 5, not helpful....",1.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1611,"[84, 3, 85, 12, 51, 9]",307,(((9+85)+12)+((3*84)-51)),Please reason and answer the following questio...,6,countdown,"<think>\nOkay, let's see. I need to use each o...","[<think>\nOkay, let's see. I need to use each ...",0.5,1.0,"[75, 96, 96, 56, 68]",129,"<think>\nOkay, let's see. I need to use each o...",<|im_start|>system\nYou are a helpful assistan...,"Therefore, the equation is 96 - 75 + 96 + 68 ...",1.0,0.0
1612,"[84, 3, 85, 12, 51, 9]",307,(((9+85)+12)+((3*84)-51)),Please reason and answer the following questio...,6,countdown,"<think>\nOkay, let's see. I need to use each o...","[<think>\nOkay, let's see. I need to use each ...",0.75,0.25,"[74, 51, 64, 31, 48]",104,"<think>\nOkay, let's see. I need to use each o...",<|im_start|>system\nYou are a helpful assistan...,Maybe (64 - 51) = 13. Then 13 + 48 = 61. Then...,1.0,0.0
1613,"[84, 3, 85, 12, 51, 9]",307,(((9+85)+12)+((3*84)-51)),Please reason and answer the following questio...,6,countdown,"<think>\nOkay, let's see. I need to use each o...","[<think>\nOkay, let's see. I need to use each ...",0.75,0.5,"[71, 93, 76, 10, 96]",184,"<think>\nOkay, let's see. I need to use each o...",<|im_start|>system\nYou are a helpful assistan...,204 is not 184. What if I do 76 + 93 = 169. 16...,0.0,0.0
1614,"[84, 3, 85, 12, 51, 9]",307,(((9+85)+12)+((3*84)-51)),Please reason and answer the following questio...,6,countdown,"<think>\nOkay, let's see. I need to use each o...","[<think>\nOkay, let's see. I need to use each ...",0.75,0.75,"[17, 76, 59, 23, 18]",41,"<think>\nOkay, let's see. I need to use each o...",<|im_start|>system\nYou are a helpful assistan

In [802]:
idx = 7
example = df.iloc[idx]
prefix_ratio, distractor_ratio = example["prefix_ratio"], example["distractor_ratio"]
print("prefix_ratio: ", prefix_ratio)
print("distractor_ratio: ", distractor_ratio)
prompt, response, original_correct, distractor_correct = example["prompt"], example["post_distraction_response"], example["original_correct"], example["distractor_correct"]

print("original correct?: ", original_correct)
print("distract correct?: ", distractor_correct)

prefix_ratio:  0.25
distractor_ratio:  1.0
original correct?:  0.0
distract correct?:  1.0


In [803]:
print(prompt)

<|im_start|>system
You are a helpful assistant.<|im_end|>
<|im_start|>user
Please reason and answer the following question. Using the numbers [50, 77, 64, 67, 73], create an equation that equals 331. You can only use basic arithmetic operations (+, -, *, /) in the expression and each number should be used exactly once. You should report the answer equation expression in <answer> </answer> tags, for example <answer> (1 + 2) / 3 </answer>.<|im_end|>
<|im_start|>assistant
<think>
Okay, let's see. I need to use each of the numbers 50, 77, 64, 67, and 73 exactly once with basic operations to make 331. Hmm. Let me think about possible combinations.Wait. Let me think again: Okay, let's see. I need to use the numbers 86, 91, 87, 68, 77, and 66 exactly once each, with basic arithmetic operations to make 165. Hmm, first, maybe I should think about possible combinations. Let me list the numbers again: 86, 91, 87, 68, 77, 66. All have to be used once. The target is 165.What's 165? Let me think abo

In [804]:
print(response)

 Which is the same as the previous grouping. Alternatively, written as (66 + 77) + (86 - 68) + (91 - 87). Either way, the operations are all allowed. Let me check again if the order is correct. Let me compute the expression step by step. 66 + 77 is 143. Then 143 + 86 is 229. Then subtract 68: 229 - 68 = 161. Then add 91: 161 + 91 = 252. Then subtract 87: 252 - 87 = 165. Correct. So that's the right equation.
</think>

<answer> 66 + 77 + 86 - 68 + 91 - 87 </answer>



In [ ]:
DataProto(batch=TensorDict(
    fields={
        attention_mask: Tensor(shape=torch.Size([6, 256]), device=cpu, dtype=torch.int64, is_shared=False),
        input_ids: Tensor(shape=torch.Size([6, 256]), device=cpu, dtype=torch.int64, is_shared=False),
        position_ids: Tensor(shape=torch.Size([6, 256]), device=cpu, dtype=torch.int64, is_shared=False)},
    batch_size=torch.Size([6]),
    device=None,
    is_shared=False), non_tensor_batch={}, meta_info={})

In [ ]:
max_prompt_length = max([len(tokenizer.encode(seq)) for seq in corrupt_sequences])

In [ ]:
ipdb> batch
DataProto(batch=TensorDict(
    fields={
        attention_mask: Tensor(shape=torch.Size([6, 256]), device=cpu, dtype=torch.int64, is_shared=False),
        input_ids: Tensor(shape=torch.Size([6, 256]), device=cpu, dtype=torch.int64, is_shared=False),
        position_ids: Tensor(shape=torch.Size([6, 256]), device=cpu, dtype=torch.int64, is_shared=False)},
    batch_size=torch.Size([6]),
    device=None,
    is_shared=False), non_tensor_batch={'data_source': array(['countdown', 'countdown', 'countdown', 'countdown', 'countdown',
       'countdown'], dtype=object), 'ability': array(['math', 'math', 'math', 'math', 'math', 'math'], dtype=object), 'reward_model': array([{'ground_truth': {'numbers': array([ 9, 48, 98, 52, 39]), 'target': 46}, 'style': 'rule'},
       {'ground_truth': {'numbers': array([54,  5,  8, 25, 14]), 'target': 90}, 'style': 'rule'},
       {'ground_truth': {'numbers': array([15, 10, 59, 96,  1]), 'target': 800}, 'style': 'rule'},
       {'ground_truth': {'numbers': array([ 8,  1,  1, 29]), 'target': 290}, 'style': 'rule'},
       {'ground_truth': {'numbers': array([71, 85,  9, 79, 40]), 'target': 49}, 'style': 'rule'},
       {'ground_truth': {'numbers': array([32, 21, 50, 45]), 'target': 42}, 'style': 'rule'}],
      dtype=object), 'extra_info': array([{'index': 15845, 'level': '5', 'split': 'train'},
       {'index': 10294, 'level': '5', 'split': 'train'},
       {'index': 19532, 'level': '5', 'split': 'train'},
       {'index': 4256, 'level': '4', 'split': 'train'},
       {'index': 11363, 'level': '5', 'split': 'train'},
       {'index': 7923, 'level': '4', 'split': 'train'}], dtype=object), 'index': array([15845, 10294, 19532, 4256, 11363, 7923], dtype=object)}, meta_info={})

In [ ]:
DataProtoItem(batch=TensorDict(
    fields={
        attention_mask: Tensor(shape=torch.Size([11, 4352]), device=cpu, dtype=torch.int64, is_shared=False),
        input_ids: Tensor(shape=torch.Size([11, 4352]), device=cpu, dtype=torch.int64, is_shared=False),
        old_log_probs: Tensor(shape=torch.Size([11, 4096]), device=cpu, dtype=torch.float32, is_shared=False),
        position_ids: Tensor(shape=torch.Size([11, 4352]), device=cpu, dtype=torch.int64, is_shared=False),
        prompts: Tensor(shape=torch.Size([11, 256]), device=cpu, dtype=torch.int64, is_shared=False),
        ref_log_prob: Tensor(shape=torch.Size([11, 4096]), device=cpu, dtype=torch.float32, is_shared=False),
        responses: Tensor(shape=torch.Size([11, 4096]), device=cpu, dtype=torch.int64, is_shared=False),
        token_level_scores: Tensor(shape=torch.Size([11, 4096]), device=cpu, dtype=torch.float32, is_shared=False)},
    batch_size=torch.Size([11]),
    device=None,
    is_shared=False), non_tensor_batch={'data_source': array(['countdown', 'countdown', 'countdown', 'countdown', 'countdown',
       'countdown', 'countdown', 'countdown', 'countdown', 'countdown',
       'countdown'], dtype=object), 'ability': array(['math', 'math', 'math', 'math', 'math', 'math', 'math', 'math',
       'math', 'math', 'math'], dtype=object), 'reward_model': array([{'ground_truth': {'numbers': array([ 9, 48, 98, 52, 39]), 'target': 46}, 'style': 'rule'},
       {'ground_truth': {'numbers': array([ 9, 48, 98, 52, 39]), 'target': 46}, 'style': 'rule'},
       {'ground_truth': {'numbers': array([ 9, 48, 98, 52, 39]), 'target': 46}, 'style': 'rule'},
       {'ground_truth': {'numbers': array([54,  5,  8, 25, 14]), 'target': 90}, 'style': 'rule'},
       {'ground_truth': {'numbers': array([54,  5,  8, 25, 14]), 'target': 90}, 'style': 'rule'},
       {'ground_truth': {'numbers': array([ 8,  1,  1, 29]), 'target': 290}, 'style': 'rule'},
       {'ground_truth': {'numbers': array([ 8,  1,  1, 29]), 'target': 290}, 'style': 'rule'},
       {'ground_truth': {'numbers': array([ 8,  1,  1, 29]), 'target': 290}, 'style': 'rule'},
       {'ground_truth': {'numbers': array([32, 21, 50, 45]), 'target': 42}, 'style': 'rule'},
       {'ground_truth': {'numbers': array([32, 21, 50, 45]), 'target': 42}, 'style': 'rule'},
       {'ground_truth': {'numbers': array([32, 21, 50, 45]), 'target': 42}, 'style': 'rule'}],
      dtype=object), 'extra_info': array([{'index': 15845, 'level': '5', 'split': 'train'},
       {'index': 15845, 'level': '5', 'split': 'train'},
       {'index': 15845, 'level': '5', 'split': 'train'},
       {'index': 10294, 'level': '5', 'split': 'train'},
       {'index': 10294, 'level': '5', 'split': 'train'},
       {'index': 4256, 'level': '4', 'split': 'train'},
       {'index': 4256, 'level': '4', 'split': 'train'},
       {'index': 4256, 'level': '4', 'split': 'train'},
       {'index': 7923, 'level': '4', 'split': 'train'},
       {'index': 7923, 'level': '4', 'split': 'train'},
       {'index': 7923, 'level': '4', 'split': 'train'}], dtype=object), 'index': array([15845, 15845, 15845, 10294, 10294, 4256, 4256, 4256, 7923, 7923,
       7923], dtype=object), 'uid': array(['3d15d528-35bf-4843-a99c-75efd44ed60c',
       '3d15d528-35bf-4843-a99c-75efd44ed60c',
       '3d15d528-35bf-4843-a99c-75efd44ed60c',
       '93f23ffe-97b6-4aa2-8523-1d094d7d5eff',
       '93f23ffe-97b6-4aa2-8523-1d094d7d5eff',
       '593cb8cb-2d8b-406e-ac4d-420403dc0547',
       '593cb8cb-2d8b-406e-ac4d-420403dc0547',
       '593cb8cb-2d8b-406e-ac4d-420403dc0547',
       'c90eb802-85c3-4f6d-9f04-09a370e618be',
       'c90eb802-85c3-4f6d-9f04-09a370e618be',
       'c90eb802-85c3-4f6d-9f04-09a370e618be'], dtype=object)}, meta_info={'global_token_num': [2417, 3337, 3542, 1948, 3350, 4211, 4213, 4213, 4213, 808, 887, 1419, 4212, 4212, 4212, 1409, 1520, 2772], 'temperature': 1.0})

In [ ]:
{'input_ids': tensor([[151643, 151643, 151643,  ..., 151644,  77091,    198],
        [151643, 151643, 151643,  ..., 151644,  77091,    198],
        [151643, 151643, 151643,  ..., 151644,  77091,    198],
        [151643, 151643, 151643,  ..., 151644,  77091,    198],
        [151643, 151643, 151643,  ..., 151644,  77091,    198],
        [151643, 151643, 151643,  ..., 151644,  77091,    198]]), 'attention_mask': tensor([[0, 0, 0,  ..., 1, 1, 1],
        [0, 0, 0,  ..., 1, 1, 1],
        [0, 0, 0,  ..., 1, 1, 1],
        [0, 0, 0,  ..., 1, 1, 1],
        [0, 0, 0,  ..., 1, 1, 1],
        [0, 0, 0,  ..., 1, 1, 1]]), 'position_ids': tensor([[  0,   0,   0,  ..., 113, 114, 115],
        [  0,   0,   0,  ..., 112, 113, 114],
        [  0,   0,   0,  ..., 114, 115, 116],
        [  0,   0,   0,  ..., 108, 109, 110],
        [  0,   0,   0,  ..., 113, 114, 115],
        [  0,   0,   0,  ..., 110, 111, 112]]), 'data_source': array(['countdown', 'countdown', 'countdown', 'countdown', 'countdown',
       'countdown'], dtype=object), 'ability': array(['math', 'math', 'math', 'math', 'math', 'math'], dtype=object), 'reward_model': array([{'ground_truth': {'numbers': array([ 9, 48, 98, 52, 39]), 'target': 46}, 'style': 'rule'},
       {'ground_truth': {'numbers': array([54,  5,  8, 25, 14]), 'target': 90}, 'style': 'rule'},
       {'ground_truth': {'numbers': array([15, 10, 59, 96,  1]), 'target': 800}, 'style': 'rule'},
       {'ground_truth': {'numbers': array([ 8,  1,  1, 29]), 'target': 290}, 'style': 'rule'},
       {'ground_truth': {'numbers': array([71, 85,  9, 79, 40]), 'target': 49}, 'style': 'rule'},
       {'ground_truth': {'numbers': array([32, 21, 50, 45]), 'target': 42}, 'style': 'rule'}],
      dtype=object), 'extra_info': array([{'index': 15845, 'level': '5', 'split': 'train'},
       {'index': 10294, 'level': '5', 'split': 'train'},
       {'index': 19532, 'level': '5', 'split': 'train'},
       {'index': 4256, 'level': '4', 'split': 'train'},
       {'index': 11363, 'level': '5', 'split': 'train'},
       {'index': 7923, 'level': '4', 'split': 'train'}], dtype=object), 'index': array([15845, 10294, 19532, 4256, 11363, 7923], dtype=object)}

In [ ]:
ipdb> gen_batch
DataProto(batch=TensorDict(
    fields={
        attention_mask: Tensor(shape=torch.Size([6, 256]), device=cpu, dtype=torch.int64, is_shared=False),
        input_ids: Tensor(shape=torch.Size([6, 256]), device=cpu, dtype=torch.int64, is_shared=False),
        position_ids: Tensor(shape=torch.Size([6, 256]), device=cpu, dtype=torch.int64, is_shared=False)},
    batch_size=torch.Size([6]),
    device=None,
    is_shared=False), non_tensor_batch={}, meta_info={})
ipdb> batch
DataProto(batch=TensorDict(
    fields={
    },
    batch_size=torch.Size([6]),
    device=None,
    is_shared=False), non_tensor_batch={'data_source': array(['countdown', 'countdown', 'countdown', 'countdown', 'countdown',
       'countdown'], dtype=object), 'ability': array(['math', 'math', 'math', 'math', 'math', 'math'], dtype=object), 'reward_model': array([{'ground_truth': {'numbers': array([ 9, 48, 98, 52, 39]), 'target': 46}, 'style': 'rule'},
       {'ground_truth': {'numbers': array([54,  5,  8, 25, 14]), 'target': 90}, 'style': 'rule'},
       {'ground_truth': {'numbers': array([15, 10, 59, 96,  1]), 'target': 800}, 'style': 'rule'},
       {'ground_truth': {'numbers': array([ 8,  1,  1, 29]), 'target': 290}, 'style': 'rule'},
       {'ground_truth': {'numbers': array([71, 85,  9, 79, 40]), 'target': 49}, 'style': 'rule'},
       {'ground_truth': {'numbers': array([32, 21, 50, 45]), 'target': 42}, 'style': 'rule'}],
      dtype=object), 'extra_info': array([{'index': 15845, 'level': '5', 'split': 'train'},
       {'index': 10294, 'level': '5', 'split': 'train'},
       {'index': 19532, 'level': '5', 'split': 'train'},
       {'index': 4256, 'level': '4', 'split': 'train'},
       {'index': 11363, 'level': '5', 'split': 'train'},
       {'index': 7923, 'level': '4', 'split': 'train'}], dtype=object), 'index': array([15845, 10294, 19532, 4256, 11363, 7923], dtype=object)}, meta_info={})

In [ ]:
corrupt_batch = batch[correct_mask]
corrupt_batch.batch.pop(batch_keys = ["input_ids", "attention_mask", "position_ids"])

In [ ]:
gen_batch_output.batch["responses"][:, -100:]

In [ ]:
batch: DataProto = DataProto.from_single_dict(batch_dict)
                # pop those keys for generation
                gen_batch = batch.pop(batch_keys=['input_ids', 'attention_mask', 'position_ids'])

                score_records = []
                with _timer('step', timing_raw):
                    # generate a batch
                    with _timer('gen', timing_raw):
                        gen_batch_output = self.actor_rollout_wg.generate_sequences(gen_batch)

                    if self.config.algorithm.adv_estimator == 'remax':
                        with _timer('gen_max', timing_raw):
                            gen_baseline_batch = deepcopy(gen_batch)
                            gen_baseline_batch.meta_info['do_sample'] = False
                            gen_baseline_output = self.actor_rollout_wg.generate_sequences(gen_baseline_batch)

                            batch = batch.union(gen_baseline_output)
                            reward_baseline_tensor, score_record = self.reward_fn(batch)
                            reward_baseline_tensor = reward_baseline_tensor.sum(dim=-1)

                            batch.pop(batch_keys=list(gen_baseline_output.batch.keys()))

                            batch.batch['reward_baselines'] = reward_baseline_tensor

                            del gen_baseline_batch, gen_baseline_output

                    batch.non_tensor_batch['uid'] = np.array([str(uuid.uuid4()) for _ in range(len(batch.batch))],
                                                             dtype=object)
                    # repeat to align with repeated responses in rollout
                    batch = batch.repeat(repeat_times=self.config.actor_rollout_ref.rollout.n, interleave=True)
                    batch = batch.union(gen_batch_output)
                    
                    # TODO
                    # HACK: Record total tokens
                    responses_length = batch.batch['responses'].size(-1)
                    self.global_total_tokens += batch.batch['attention_mask'][:, -responses_length:].sum().item()
                    metrics.update({'token_budget/total_token_budget': self.global_total_tokens})
                    # End of HACK

                    # balance the number of valid tokens on each dp rank.
                    # Note that this breaks the order of data inside the batch.
                    # Please take care when you implement group based adv computation such as GRPO and rloo
                    self._balance_batch(batch, metrics=metrics)

                    # compute global_valid tokens
                    batch.meta_info['global_token_num'] = torch.sum(batch.batch['attention_mask'], dim=-1).tolist()

                    # recompute old_log_probs
                    with _timer('old_log_prob', timing_raw):
                        old_log_prob = self.actor_rollout_wg.compute_log_prob(batch)
                        batch = batch.union(old_log_prob)

                    if self.use_reference_policy:
                        # compute reference log_prob
                        with _timer('ref', timing_raw):
                            ref_log_prob = self.ref_policy_wg.compute_ref_log_prob(batch)
                            batch = batch.union(ref_log_prob)

                    # compute values
                    if self.use_critic:
                        with _timer('values', timing_raw):
                            values = self.critic_wg.compute_values(batch)
                            batch = batch.union(values)

                    with _timer('adv', timing_raw):
                        # compute scores. Support both model and function-based.
                        # We first compute the scores using reward model. Then, we call reward_fn to combine
                        # the results from reward model and rule-based results.
                        if self.use_rm:
                            # we first compute reward model score
                            reward_tensor = self.rm_wg.compute_rm_score(batch)
                            batch = batch.union(reward_tensor)

                        # we combine with rule-based rm
                        reward_tensor, score_record = self.reward_fn(batch)
                        score_records.extend(score_record)
                        batch.batch['token_level_scores'] = reward_tensor

                        #HACK: Update reasoning dataset
                        self.update_rollout_dataset(batch, reward_tensor, score_record)
                        
                        # HACK: Record # solve none and solve all and avg solve rate
                        uids = batch.non_tensor_batch['uid']
                        unique_uids = np.unique(uids)
                        valid_mask = torch.ones(len(uids), dtype=torch.bool)
                        solve_none = 0
                        solve_all = 0

                        for uid in unique_uids:
                            uid_mask = uids == uid
                            uid_rewards = reward_tensor[uid_mask].sum(-1)  # Sum rewards for each sequence
                            
                            # Check if all rewards are 0 or all are 1 for this uid
                            if (uid_rewards == 0).all():
                                valid_mask[uid_mask] = False
                                solve_none += 1
                            elif (uid_rewards == 1).all():
                                valid_mask[uid_mask] = False
                                solve_all += 1
                        
                        # Log to metrics
                        metrics['batch/solve_none'] = solve_none
                        metrics['batch/solve_all'] = solve_all
                        metrics['batch/avg_solve_rate'] = batch[valid_mask].batch['token_level_scores'].sum(-1).mean().item()

In [ ]:
                    if not self.config.actor_rollout_ref.actor.get('use_kl_loss', False):
                        batch, kl_metrics = apply_kl_penalty(batch,
                                                             kl_ctrl=self.kl_ctrl,
                                                             kl_penalty=self.config.algorithm.kl_penalty)
                        metrics.update(kl_metrics)
                    else:
                        batch.batch['token_level_rewards'] = batch.batch['token_level_scores']

                    # compute advantages, executed on the driver process
                    batch = compute_advantage(batch,
                                              adv_estimator=self.config.algorithm.adv_estimator,
                                              gamma=self.config.algorithm.gamma,
                                              lam=self.config.algorithm.lam,
                                              num_repeat=self.config.actor_rollout_ref.rollout.n)

                # update critic
                if self.use_critic:
                    with _timer('update_critic', timing_raw):
                        critic_output = self.critic_wg.update_critic(batch)
                    critic_output_metrics = reduce_metrics(critic_output.meta_info['metrics'])
                    metrics.update(critic_output_metrics)

                # implement critic warmup
                if self.config.trainer.critic_warmup <= self.global_steps:
                    # update actor
                    with _timer('update_actor', timing_raw):
                        actor_output = self.actor_rollout_wg.update_actor(batch)
                    actor_output_metrics = reduce_metrics(actor_output.meta_info['metrics'])
                    metrics.update(actor_output_metrics)

                with _timer('log_scores', timing_raw):
                    score_metrics = self._log_scores_to_wandb(score_records, epoch)
                    # Update metrics with returned values instead of direct logging
                    # logger.log(data=score_metrics, step=self.global_steps)
                    metrics.update(score_metrics)


In [ ]:
ipdb> output[1]
tensor([[-1.2297e-03, -1.1921e-07, -2.0623e-05,  ...,  1.5164e+05,
          1.5164e+05,  1.5164e+05],
        [-1.2297e-03, -1.1921e-07, -2.0623e-05,  ...,  1.5164e+05,
          1.5164e+05,  1.5164e+05],
        [-1.2297e-03, -1.1921e-07, -2.0623e-05,  ...,  1.5164e+05,
          1.5164e+05,  1.5164e+05],
        ...,
        [-1.3803e-03, -1.1921e-07, -2.0623e-05,  ...,  1.5164e+05,
          1.5164e+05,  1.5164e+05],
        [-1.3803e-03, -1.1921e-07, -2.0623e-05,  ...,  1.5164e+05,
          1.5164e+05,  1.5164e+05],
        [-1.3803e-03, -1.1921e-07, -2.0623e-05,  ...,  1.5164e+05,
          1.5164e+05,  1.5164e+05]])
ipdb> output[0]
tensor([[ 13708,    766,    397,  ..., 151643, 151643, 151643],
        [ 13708,    766,    397,  ..., 151643, 151643, 151643],
        [ 13708,    766,    397,  ..., 151643, 151643, 151643],
        ...,
        [ 13708,    766,    397,  ..., 151643, 151643, 151643],
        [ 13708,    766,    397,  ..., 151643, 151643, 151643],
        [ 13708,    766,    397,  ..., 151643, 151643, 151643]])
ipdb> output[0].shape
torch.Size([18, 4096])
ipdb> output[1]
tensor([[-1.2297e-03, -1.1921e-07, -2.0623e-05,  ...,  1.5164e+05,
          1.5164e+05,  1.5164e+05],
        [-1.2297e-03, -1.1921e-07, -2.0623e-05,  ...,  1.5164e+05,
          1.5164e+05,  1.5164e+05],
        [-1.2297e-03, -1.1921e-07, -2.0623e-05,  ...,  1.5164e+05,
          1.5164e+05,  1.5164e+05],
        ...,
        [-1.3803e-03, -1.1921e-07, -2.0623e-05,  ...,  1.5164e+05,
          1.5164e+05,  1.5164e+05],
        [-1.3803e-03, -1.1921e-07, -2.0623e-05,  ...,  1.5164e+05,
          1.5164e+05,  1.5164e+05],
        [-1.3803e-03, -1.1921e-07, -2.0623e-05,  ...,  1.5164e+05,
          1.5164e+05,  1.5164e+05]])
ipdb> output[1].shape
torch.Size([18, 4096])
ipdb>

In [ ]:
evals
    0. same set of stress tests for both math and countdown
    1. if observations on countdown and math are consistent
    2. if the corruption happens in the first chunk
    3. check the eval results for sft 3 epoch

In [ ]:
training todo:
    
    mix original question and corrupted prefix
    1. collect offline rl prefix ==> rl (save checkpoints for later eval)
    2. rl on same flops ==> same number of problems (assuming problems i.i.d)
    3. 1st rl collect for rejection sampling
    
    
generalization:
    1. different types of corruption (distractor injection, just wrong answer at the end)
    2. different degrees of digit corruption
    3. different task complexity
    


In [ ]:
generate more level 5, 6, 7 questions 50K each
use r1 to generate corrupted prefix + collect recovery + not recover (50 50)

we can launch 2 first